# UC: Aprendizagem Profunda - Trabalho Prático
## Tarefa 4: Avaliação dos Modelos e Pipeline Oficial de Submissão

**Docentes:** Miguel Rocha, Lázaro Makili  
**Objetivo:** Distinguir entre texto gerado por Inteligência Artificial (AI - 1) e texto escrito por Humanos (Human - 0).

---

### Organização das Submissões:
1. **1ª Submissão:** Dois melhores modelos por grupo (sendo **um de implementação própria em NumPy** e um em PyTorch).
2. **2ª Submissão:** Dois melhores modelos globais (PyTorch BiLSTM / PyTorch DNN).

Os modelos estão **pré-treinados e guardados** na diretoria `models/`, permitindo inferência direta e imediata sobre qualquer conjunto de validação externa fornecido pelos docentes.

In [ ]:
# 1. Configuração do Ambiente e Importação de Módulos
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Adicionar src ao path
sys.path.insert(0, os.path.abspath("../src"))
sys.path.insert(0, os.path.abspath("src"))

from inference import ModelInferenceEngine

# Inicializar o motor de inferência (carrega pesos e pré-processadores da pasta 'models/')
engine = ModelInferenceEngine(models_dir="../models" if os.path.exists("../models") else "models")
print("[*] Motor de Inferência inicializado com sucesso!")
print(f"[*] Dispositivo ativo: {engine.device}")

--- 
## 2. Benchmark Comparativo e Resultados Obtidos

Abaixo apresentam-se os resultados do benchmark não-enviesado no conjunto de teste e as visualizações comparativas das 5 arquiteturas desenvolvidas:

In [ ]:
# Carregar a tabela de benchmark gerada
results_table_path = os.path.join("..", "results", "benchmark_ranking_table.csv")
if not os.path.exists(results_table_path):
    results_table_path = os.path.join("results", "benchmark_ranking_table.csv")

if os.path.exists(results_table_path):
    df_bench = pd.read_csv(results_table_path)
    display_cols = ['Rank', 'Model_Name', 'Framework', 'Accuracy', 'F1_Score', 'Precision', 'Recall', 'ROC_AUC', 'Latency_ms_per_sample']
    df_display = df_bench[display_cols].copy()
    df_display['Accuracy'] = df_display['Accuracy'].apply(lambda x: f"{x*100:.2f}%")
    df_display['F1_Score'] = df_display['F1_Score'].apply(lambda x: f"{x*100:.2f}%")
    df_display['Precision'] = df_display['Precision'].apply(lambda x: f"{x*100:.2f}%")
    df_display['Recall'] = df_display['Recall'].apply(lambda x: f"{x*100:.2f}%")
    df_display['Latency_ms_per_sample'] = df_display['Latency_ms_per_sample'].apply(lambda x: f"{x:.2f} ms")
    display(df_display)
else:
    print("Tabela de benchmark ainda não gerada. Execute tarefa4_avaliacao_modelos.py")

### Gráficos de Avaliação Comparativa

In [ ]:
from IPython.display import Image, display

res_dir = "../results" if os.path.exists("../results") else "results"
images = [
    os.path.join(res_dir, "benchmark_metrics_comparison.png"),
    os.path.join(res_dir, "roc_curves_all_models.png"),
    os.path.join(res_dir, "confusion_matrices_all_models.png"),
    os.path.join(res_dir, "latency_vs_accuracy.png")
]

for img_path in images:
    if os.path.exists(img_path):
        display(Image(filename=img_path))
        print("-" * 80)

---
## 3. Função Universal de Submissão

Esta função permite aos docentes correrem qualquer dataset novo (`.csv` ou `.tsv` com colunas `ID` e `text`) e gerar automaticamente as previsões no formato padrão exigido.

In [ ]:
def executar_submissao_docente(input_file_path, output_file_path, model_choice="pytorch_bilstm"):
    """
    Executa a inferência e grava o ficheiro de submissão.
    
    Opções para model_choice:
      - 'numpy_lr': Regressão Logística NumPy (1ª Submissão - Própria)
      - 'numpy_dnn': Rede Neuronal Profunda NumPy (1ª Submissão - Própria)
      - 'pytorch_bilstm': PyTorch BiLSTM com Embeddings (Melhor F1-Score)
      - 'pytorch_dnn': PyTorch Tabular DNN com TF-IDF
      - 'pytorch_transformer': PyTorch Transformer Encoder
    """
    print(f"\n{'='*70}")
    print(f"[*] A PROCESSAR SUBMISSÃO COM MODELO: {model_choice}")
    print(f"[*] Ficheiro de Entrada: {input_file_path}")
    print(f"[*] Ficheiro de Saída  : {output_file_path}")
    print(f"{'='*70}")
    
    df_out = engine.generate_submission(
        input_csv_path=input_file_path,
        output_csv_path=output_file_path,
        model_type=model_choice
    )
    return df_out

---
## 4. Demonstração das Submissões (Fase 1 e Fase 2)

Abaixo corre-se a validação sobre o ficheiro simulado do docente (`data/simulated_teacher_validation.csv` ou qualquer ficheiro fornecido):

In [ ]:
# Ficheiro de validação de exemplo (textos 100-120 palavras em Ciências Naturais e Engenharia)
sample_input = "../data/simulated_teacher_validation.csv" if os.path.exists("../data/simulated_teacher_validation.csv") else "data/simulated_teacher_validation.csv"

# -------------------------------------------------------------
# 4.1 SUBMISSÃO FASE 1: Modelo NumPy de Raiz (Próprio) + Modelo PyTorch
# -------------------------------------------------------------
print("\n>>> GERANDO SUBMISSÃO 1A: Modelo NumPy de Raiz (Regressão Logística/DNN)")
df_sub_fase1_numpy = executar_submissao_docente(
    input_file_path=sample_input,
    output_file_path="submissao_fase1_numpy.csv",
    model_choice="numpy_lr"
)
display(df_sub_fase1_numpy.head())

print("\n>>> GERANDO SUBMISSÃO 1B: Modelo PyTorch (BiLSTM com Embeddings)")
df_sub_fase1_pytorch = executar_submissao_docente(
    input_file_path=sample_input,
    output_file_path="submissao_fase1_pytorch.csv",
    model_choice="pytorch_bilstm"
)
display(df_sub_fase1_pytorch.head())

# -------------------------------------------------------------
# 4.2 SUBMISSÃO FASE 2: Dois Melhores Modelos Globais
# -------------------------------------------------------------
print("\n>>> GERANDO SUBMISSÃO 2A: Melhor Modelo Global (PyTorch BiLSTM)")
df_sub_fase2_top1 = executar_submissao_docente(
    input_file_path=sample_input,
    output_file_path="submissao_fase2_top1.csv",
    model_choice="pytorch_bilstm"
)

print("\n>>> GERANDO SUBMISSÃO 2B: Segundo Melhor Modelo Global (PyTorch Tabular DNN)")
df_sub_fase2_top2 = executar_submissao_docente(
    input_file_path=sample_input,
    output_file_path="submissao_fase2_top2.csv",
    model_choice="pytorch_dnn"
)
display(df_sub_fase2_top2.head())

---
## 5. Teste Interativo com Novo Texto em Tempo Real

Pode testar qualquer texto inserido manualmente para obter a classificação imediata de IA vs Humano:

In [ ]:
texto_exemplo = """
In this study, we investigated the thermodynamic properties of superconducting materials at low temperatures.
Our empirical results demonstrated a significant phase transition with zero electrical resistance below 90 Kelvin.
"""

resultado = engine.predict_single_text(texto_exemplo, model_type="pytorch_bilstm")

print("Resultado da Predição:")
print(f"  - Classificação : {resultado['predicted_label']} (Código: {resultado['label_num']})")
print(f"  - Probabilidade IA: {resultado['probability_ai']*100:.2f}%")
print(f"  - Probabilidade Humano: {resultado['probability_human']*100:.2f}%")
print(f"  - Contagem de Palavras: {resultado['word_count']}")
print(f"  - Modelo Utilizado: {resultado['model_used']}")